Voici le notebook d'application de la théorie des valeurs extrêmes sur le jeu de données NBA.

## 1 Import/Initialisation 

Je reprends ici principalement le code du nba_api.ipynb afin de définir les différentes fonctions d'import etc

In [ ]:
!pip install nba_api scipy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import time
from scipy.stats import genextreme, genpareto


from nba_api.stats.endpoints import (
    leaguegamefinder,
    boxscoretraditionalv2,
    playbyplayv3,
    teamgamelog,
)
from nba_api.stats.static import teams, players

from utils import nba_request_with_retry, get_team_id ,get_box, get_extended_pbp, analysis_extreme_run, fit_gev_distribution, get_pbp, get_games_played, point_filter1, point_filter2,fetch_pbp_for_games, SLEEP

### Données anecdotiques

In [3]:
# Toutes les équipes NBA
all_teams = teams.get_teams()
df_teams = pd.DataFrame(all_teams)
print(f"Nombre d'équipes : {len(df_teams)}")
df_teams.head(10)

Nombre d'équipes : 30


,id,full_name,abbreviation,nickname,city,state,year_founded
0,1610612737,Atlanta Hawks,ATL,Hawks,Atlanta,Georgia,1949
1,1610612738,Boston Celtics,BOS,Celtics,Boston,Massachusetts,1946
2,1610612739,Cleveland Cavaliers,CLE,Cavaliers,Cleveland,Ohio,1970
3,1610612740,New Orleans Pelicans,NOP,Pelicans,New Orleans,Louisiana,2002
4,1610612741,Chicago Bulls,CHI,Bulls,Chicago,Illinois,1966
5,1610612742,Dallas Mavericks,DAL,Mavericks,Dallas,Texas,1980
6,1610612743,Denver Nuggets,DEN,Nuggets,Denver,Colorado,1976
7,1610612744,Golden State Warriors,GSW,Warriors,San Francisco,California,1946
8,1610612745,Houston Rockets,HOU,Rockets,Houston,Texas,1967
9,1610612746,Los Angeles Clippers,LAC,Clippers,Los Angeles,California,1970


In [3]:
gsw = get_team_id("Golden State Warriors")
bos = get_team_id("Boston Celtics")
print("Golden State Warriors :", gsw)
print("Boston Celtics        :", bos)

Golden State Warriors : {'id': 1610612744, 'full_name': 'Golden State Warriors', 'abbreviation': 'GSW', 'nickname': 'Warriors', 'city': 'San Francisco', 'state': 'California', 'year_founded': 1946}
Boston Celtics        : {'id': 1610612738, 'full_name': 'Boston Celtics', 'abbreviation': 'BOS', 'nickname': 'Celtics', 'city': 'Boston', 'state': 'Massachusetts', 'year_founded': 1946}


In [4]:
SEASON = "2023-24"
TEAM_ID = gsw["id"]  # Golden State Warriors

df_games = get_games_played(TEAM_ID, SEASON)
print(f"Matchs récupérés : {len(df_games)}")
df_games[["Game_ID", "GAME_DATE", "MATCHUP", "WL", "PTS"]].head(10)

Tentative 1/4 échouée (ReadTimeout). Nouvelle tentative dans 3s…


KeyboardInterrupt: 

In [ ]:
# Sélectionner un match précis pour la suite (1er match de la liste)
GAME_ID = df_games["Game_ID"].iloc[0]
print(f"Match sélectionné : {df_games['MATCHUP'].iloc[0]}  —  {df_games['GAME_DATE'].iloc[0]}")
print(f"Game ID           : {GAME_ID}")

In [ ]:
box = get_box(game_id = GAME_ID)

df_players_box = box.get_data_frames()[0]
df_teams_box   = box.get_data_frames()[1]

cols_joueur = ["TEAM_ABBREVIATION", "PLAYER_NAME", "MIN", "PTS", "REB", "AST", "STL", "BLK", "TO", "PLUS_MINUS"]
print("=== Box Score Joueurs ===")
df_players_box[cols_joueur].dropna(subset=["MIN"])


In [ ]:
print("=== Box Score Équipes ===")
df_teams_box[["TEAM_ABBREVIATION", "PTS", "FG_PCT", "FG3_PCT", "FT_PCT", "REB", "AST", "STL", "BLK", "TO"]]

In [ ]:
df_pbp = get_pbp(game_id = GAME_ID)
df_pbp.head(10)

In [ ]:
df = get_extended_pbp(game_id = GAME_ID)

print("Aperçu enrichi :")
df[["ELAPSED_MINUTES", "period", "clock", "description",
    "SCORE_HOME", "SCORE_VISITOR", "SCOREMARGIN_FF"]].dropna(
    subset=["ELAPSED_MINUTES"]
).head(15)

In [ ]:
# Récupérer les noms des deux équipes depuis le box score
home_team = df_teams_box.iloc[0]["TEAM_ABBREVIATION"]
away_team = df_teams_box.iloc[1]["TEAM_ABBREVIATION"]

In [ ]:
runs = point_filter1(game_id = GAME_ID)

# Top 10 des plus grands runs
top_runs = runs.sort_values("pts", ascending=False).head(10)
print("Top 10 des runs du match :")
top_runs[["SCORER", "pts", "start_min", "end_min", "n_events"]].rename(
    columns={"SCORER":"Équipe","pts":"Points marqués","start_min":"Début (min)",
             "end_min":"Fin (min)","n_events":"Nb actions"}
).round(2)

## 2 Réponses aux questions

### Q1 — Quelle est la probabilité d'observer un run extrême ? (niveau brut ou niveau score pondéré)

#### Définition run extrême + visualisation des distributions

In [ ]:
import numpy as np

# Define what an 'extreme run' means. For example, runs with >= 8 points.
extreme_threshold = 8

# Filter runs that meet the extreme threshold
extreme_runs = runs[runs['pts'] >= extreme_threshold]

# Calculate the total number of runs observed
total_runs_count = len(runs)

# Calculate the number of extreme runs
extreme_runs_count = len(extreme_runs)

# Calculate the probability of an extreme run
probability_extreme_run = extreme_runs_count / total_runs_count if total_runs_count > 0 else 0

print(f"Total runs analyzed: {total_runs_count}")
print(f"Number of extreme runs (>= {extreme_threshold} points): {extreme_runs_count}")
print(f"Probability of observing an extreme run: {probability_extreme_run:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.histplot(runs['pts'], bins=20, kde=True)
plt.axvline(x=extreme_threshold, color='red', linestyle='--', label=f'Extreme Run Threshold ({extreme_threshold} pts)')
plt.title('Distribution of Run Points')
plt.xlabel('Points Scored in a Run')
plt.ylabel('Frequency')
plt.legend()
plt.grid(axis='y', alpha=0.75)
plt.show()

In [1]:
weighted_runs = point_filter2(game_id=GAME_ID)

# Top 10 of the largest weighted runs
top_weighted_runs = weighted_runs.sort_values("weighted_pts", ascending=False).head(10)
print("Top 10 weighted runs of the match:")
print(top_weighted_runs[["SCORER", "weighted_pts", "start_min", "end_min", "n_events"]].rename(
    columns={
        "SCORER": "Équipe",
        "weighted_pts": "Points pondérés",
        "start_min": "Début (min)",
        "end_min": "Fin (min)",
        "n_events": "Nb actions"
    })
.round(2))

NameError: name 'point_filter2' is not defined

In [ ]:
# Druns with >= 10 weighted points "extreme weighted runs".
extreme_weighted_threshold = 10

# Filter runs that meet the extreme threshold
extreme_weighted_runs = weighted_runs[weighted_runs['weighted_pts'] >= extreme_weighted_threshold]

total_weighted_runs_count = len(weighted_runs)
extreme_weighted_runs_count = len(extreme_weighted_runs)
probability_extreme_weighted_run = extreme_weighted_runs_count / total_weighted_runs_count if total_weighted_runs_count > 0 else 0

print(f"Total weighted runs analyzed: {total_weighted_runs_count}")
print(f"Number of extreme weighted runs (>= {extreme_weighted_threshold} weighted points): {extreme_weighted_runs_count}")
print(f"Probability of observing an extreme weighted run: {probability_extreme_weighted_run:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(weighted_runs['weighted_pts'], bins=20, kde=True)
plt.axvline(x=extreme_weighted_threshold, color='red', linestyle='--', label=f'Extreme Weighted Run Threshold ({extreme_weighted_threshold} pts)')
plt.title('Distribution of Weighted Run Points')
plt.xlabel('Weighted Points Scored in a Run')
plt.ylabel('Frequency')
plt.legend()
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
full_analysis_summary = analysis_extreme_run(SEASON, TEAM_ID)
full_analysis_summary["prob_extreme_raw_run"] = pd.to_numeric(full_analysis_summary["prob_extreme_raw_run"])
full_analysis_summary["prob_extreme_weighted_run"] = pd.to_numeric(full_analysis_summary["prob_extreme_weighted_run"])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram for probability of extreme raw runs
sns.histplot(full_analysis_summary['prob_extreme_raw_run'], bins=10, kde=True, ax=axes[0])
axes[0].set_title('Distribution of Probability of Extreme Raw Runs')
axes[0].set_xlabel('Probability')
axes[0].set_ylabel('Frequency')

# Histogram for probability of extreme weighted runs
sns.histplot(full_analysis_summary['prob_extreme_weighted_run'], bins=10, kde=True, ax=axes[1])
axes[1].set_title('Distribution of Probability of Extreme Weighted Runs')
axes[1].set_xlabel('Probability')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Extract all raw run point distributions and flatten them into a single list
all_raw_run_points = [point for sublist in full_analysis_summary['raw_run_points_distribution'] for point in sublist if sublist is not None]

# 2. Create a histogram
plt.figure(figsize=(10, 6))
sns.histplot(all_raw_run_points, bins=20, kde=True)

# 3. Set the title of the plot
plt.title('Distribution of All Raw Run Points Across All Games')

# 4. Label the x-axis and y-axis
plt.xlabel('Raw Run Points')
plt.ylabel('Frequency')

# Add grid for better readability
plt.grid(axis='y', alpha=0.75)

# 5. Display the plot
plt.show()


In [ ]:
# 1. Extract all weighted run point distributions and flatten them into a single list
all_weighted_run_points = [point for sublist in full_analysis_summary['weighted_run_points_distribution'] for point in sublist if sublist is not None]

# 2. Create a histogram
plt.figure(figsize=(10, 6))
sns.histplot(all_weighted_run_points, bins=20, kde=True)

# 3. Set the title of the plot
plt.title('Distribution of All Weighted Run Points Across All Games')

# 4. Label the x-axis and y-axis
plt.xlabel('Weighted Run Points')
plt.ylabel('Frequency')

# Add grid for better readability
plt.grid(axis='y', alpha=0.75)

# 5. Display the plot
plt.show()

In [ ]:
print(full_analysis_summary.head())

#### Fisher–Tippett–Gnedenko theorem 
According to this theorem, both distributions should be in one of the three families : Gumbel, Fréchet and Weibull families 

In [ ]:
print("Fitting GEV distribution to all raw run points...")
c_raw, loc_raw, scale_raw = fit_gev_distribution(all_raw_run_points)
print(f"GEV parameters for raw run points: Shape={c_raw:.4f}, Location={loc_raw:.4f}, Scale={scale_raw:.4f}")

print("\nFitting GEV distribution to all weighted run points...")
c_weighted, loc_weighted, scale_weighted = fit_gev_distribution(all_weighted_run_points)
print(f"GEV parameters for weighted run points: Shape={c_weighted:.4f}, Location={loc_weighted:.4f}, Scale={scale_weighted:.4f}")

The Generalized Extreme Value (GEV) distribution was fitted to `all_raw_run_points`, yielding the following parameters:
*   Shape: -0.1371
*   Location: 2.4415
*   Scale: 1.2515

The GEV distribution was also fitted to `all_weighted_run_points`, resulting in these parameters:
*   Shape: -0.1281
*   Location: 1.8223
*   Scale: 1.3704

Comparing the two distributions, the `all_weighted_run_points` show a lower location parameter (1.8223 vs 2.4415), suggesting a shift towards smaller extreme values, and a slightly higher scale parameter (1.3704 vs 1.2515), indicating a broader spread of extreme values compared to the `all_raw_run_points`.

Further analysis could involve plotting the fitted GEV distributions against the histograms of both raw and weighted run points to visually assess the goodness of fit and understand the implications of these parameter differences on the tail behavior of the data.


We will now visualize the GEV fit for both raw and weighted run points by plotting histograms and overlaying the corresponding probability density functions. Then, interpret the shape parameters of the fitted GEV distributions to determine if they resemble Fréchet, Weibull, or Gumbel distributions.

In [ ]:
plt.figure(figsize=(10, 6))

# Plot the histogram of all_raw_run_points
sns.histplot(all_raw_run_points, bins=20, kde=False, stat='density', color='skyblue', label='Raw Run Points Histogram')

# Generate x-values for plotting the PDF
x = np.linspace(min(all_raw_run_points) - 1, max(all_raw_run_points) + 1, 1000)

# Calculate the PDF values for the GEV distribution
pdf_raw = genextreme.pdf(x, c_raw, loc=loc_raw, scale=scale_raw)

plt.plot(x, pdf_raw, color='red', linestyle='--', label='Fitted GEV PDF')

plt.title('GEV Fit for Raw Run Points')
plt.xlabel('Raw Run Points')
plt.ylabel('Density')
plt.legend()
plt.grid(axis='y', alpha=0.75)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(all_weighted_run_points, bins=20, kde=False, stat='density', color='lightcoral', label='Weighted Run Points Histogram')

# Generate x-values for plotting the PDF
x_weighted = np.linspace(min(all_weighted_run_points) - 1, max(all_weighted_run_points) + 1, 1000)

# Calculate the PDF values for the GEV distribution
pdf_weighted = genextreme.pdf(x_weighted, c_weighted, loc=loc_weighted, scale=scale_weighted)

# Plot the GEV PDF curve
plt.plot(x_weighted, pdf_weighted, color='darkred', linestyle='--', label='Fitted GEV PDF')

plt.title('GEV Fit for Weighted Run Points')
plt.xlabel('Weighted Run Points')
plt.ylabel('Density')
plt.legend()
plt.grid(axis='y', alpha=0.75)

plt.show()

The Generalized Extreme Value (GEV) distribution has a shape parameter 'c' (sometimes denoted as ξ, or k in some conventions) that dictates its type:

*   **c > 0 (Fréchet distribution)**: Heavy-tailed distribution, often associated with unbounded maxima. The probability of extreme events is higher.
*   **c < 0 (Weibull distribution)**: Light-tailed distribution, associated with bounded maxima. There is an upper limit to possible extreme values.
*   **c ≈ 0 (Gumbel distribution)**: Represents an exponential-type tail behavior, falling between Fréchet and Weibull. It is often a good approximation when the true distribution is unknown or when c is close to zero.

Let's analyze the fitted shape parameters:

*   **Raw Run Points**: `c_raw` = -0.1371
    Since `c_raw` is negative (-0.1371 < 0), the distribution of raw run points resembles a **Weibull distribution**. This suggests that there might be an upper bound to the raw points scored in a run, meaning extremely high raw point runs are less likely than what a Fréchet distribution would imply.

*   **Weighted Run Points**: `c_weighted` = -0.1281
    Similarly, `c_weighted` is also negative (-0.1281 < 0). Therefore, the distribution of weighted run points also resembles a **Weibull distribution**. This implies a similar behavior to raw run points, with a potential upper limit on the maximum weighted points scored in a continuous run.

In both cases, the negative shape parameter indicates that the extreme runs (both raw and weighted) are better described by a Weibull-type distribution, suggesting that there's a practical limit to how large these runs can get.

In [ ]:
upper_limit_raw = loc_raw - scale_raw / c_raw
print(f"Theoretical upper limit for raw run points: {upper_limit_raw:.4f}")

In [ ]:
upper_limit_weighted = loc_weighted - scale_weighted / c_weighted
print(f"Theoretical upper limit for weighted run points: {upper_limit_weighted:.4f}")

The problem of our current modelisation is that we are only looking at the distribution of all runs, without distinguishing between "normal" runs and "extreme" runs. In reality, the distribution of extreme runs (e.g., runs with 8 or more points) may differ significantly from the distribution of all runs. By fitting a GEV distribution to all runs, we may be underestimating the tail behavior and thus the probability of observing extreme runs.

In [ ]:
# To address the issue of underestimating the tail behavior by fitting GEV to all runs,
# we fit the GEV distribution only to the extreme runs (e.g., runs with >= 8 points for raw, >= 10 for weighted).

extreme_threshold_raw = 8
extreme_threshold_weighted = 10

extreme_raw_run_points = [p for p in all_raw_run_points if p >= extreme_threshold_raw]
extreme_weighted_run_points = [p for p in all_weighted_run_points if p >= extreme_threshold_weighted]

print(f"Number of extreme raw runs (>= {extreme_threshold_raw}): {len(extreme_raw_run_points)}")
print(f"Number of extreme weighted runs (>= {extreme_threshold_weighted}): {len(extreme_weighted_run_points)}")

if len(extreme_raw_run_points) > 0:
    print("Fitting GEV distribution to extreme raw run points...")
    c_extreme_raw, loc_extreme_raw, scale_extreme_raw = fit_gev_distribution(extreme_raw_run_points)
    print(f"GEV parameters for extreme raw run points: Shape={c_extreme_raw:.4f}, Location={loc_extreme_raw:.4f}, Scale={scale_extreme_raw:.4f}")
else:
    print("No extreme raw runs found.")

if len(extreme_weighted_run_points) > 0:
    print("Fitting GEV distribution to extreme weighted run points...")
    c_extreme_weighted, loc_extreme_weighted, scale_extreme_weighted = fit_gev_distribution(extreme_weighted_run_points)
    print(f"GEV parameters for extreme weighted run points: Shape={c_extreme_weighted:.4f}, Location={loc_extreme_weighted:.4f}, Scale={scale_extreme_weighted:.4f}")
else:
    print("No extreme weighted runs found.")

To have another point of view of the question of distribution of the longest run, we can also explore the "Peaks Over Threshold" (POT) method using the Generalized Pareto Distribution (GPD). 

In [ ]:
threshold_90th_percentile = np.percentile(all_raw_run_points, 90)
print(f"90th percentile threshold for raw run points: {threshold_90th_percentile:.4f}")

exceedances_raw = [point for point in all_raw_run_points if point > threshold_90th_percentile]
print(f"Number of raw run points exceeding the 90th percentile: {len(exceedances_raw)}")

In [ ]:
shape_gpd, loc_gpd, scale_gpd = genpareto.fit(exceedances_raw)

print(f"Fitted GPD Parameters for Raw Run Exceedances (over {threshold_90th_percentile:.2f}):")
print(f"  Shape (c): {shape_gpd:.4f}")
print(f"  Location (loc): {loc_gpd:.4f}")
print(f"  Scale (scale): {scale_gpd:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(exceedances_raw, bins=20, kde=False, stat='density', color='skyblue', label='Raw Run Exceedances Histogram')

x_gpd = np.linspace(min(exceedances_raw) - 1, max(exceedances_raw) + 1, 1000)

pdf_gpd = genpareto.pdf(x_gpd, shape_gpd, loc=loc_gpd, scale=scale_gpd)

plt.plot(x_gpd, pdf_gpd, color='red', linestyle='--', label='Fitted GPD PDF')

plt.title('GPD Fit for Raw Run Exceedances')
plt.xlabel('Raw Run Exceedances')
plt.ylabel('Density')

plt.legend()

plt.grid(axis='y', alpha=0.75)

plt.show()

We have fitted the Generalized Pareto Distribution (GPD) to the raw run exceedances, i.e., those raw run points that exceeded the 90th percentile threshold (`threshold_90th_percentile = 6.0`). The fitted GPD parameters are:

*   **Shape (c):** `shape_gpd = 0.7642`
*   **Location (loc):** `loc_gpd = 7.0000`
*   **Scale (scale):** `scale_gpd = 0.5764`

The **shape parameter `c` ** of the GPD is crucial for understanding the tail behavior of the distribution of exceedances:

*   **If `c > 0` (as observed here, `0.7642`):** The distribution has a **heavy tail** and is unbounded above. This means that extremely large observations (raw run points) are possible, and their probability decays slowly. There is no theoretical upper limit to the raw run points when modeling the exceedances with a GPD with a positive shape parameter.
*   If `c = 0` (Exponential distribution): The tail is lighter, decaying exponentially.
*   If `c < 0`: The distribution has a finite upper bound.

Our `shape_gpd` of `0.7642` indicates a heavy-tailed distribution for the raw run exceedances. This directly addresses the observation that some raw run points exceeded the GEV's theoretical upper limit.

Previously, when fitting the GEV distribution to *all* raw run points, we found a shape parameter `c_raw = -0.1371`. A negative shape parameter in GEV implies a finite upper bound (`11.5727`). The discrepancy arises because the GEV was fitted to the entire dataset of run points, which includes many smaller values.

However, the GPD, specifically designed for modeling exceedances over a high threshold (Peaks Over Threshold method), provides a different picture for the *extreme* tail. The positive `shape_gpd` suggests that once a run crosses a certain threshold (e.g., 6 points in our case), its potential to grow even larger is not limited by a finite ceiling. This explains why we observed raw runs significantly higher than the GEV's theoretical limit (e.g., `max_observed_raw_run = 22.0`). The GEV, when applied to the full dataset, might underestimate the true potential for extreme events because it tries to fit the entire distribution, including the bulk, which can mask the true nature of the very far tail.

Thus, the GPD model with `c > 0` is more appropriate for describing the *extreme* tail behavior of raw runs, especially since it aligns with the empirical observation of runs exceeding the GEV's derived upper bound. It acknowledges the possibility of truly exceptional runs.

The **scale parameter `scale_gpd = 0.5764`** (or `σ`) indicates the spread or variability of the exceedances *above* the threshold. A larger scale parameter would imply that the exceedances are more dispersed, meaning there's a wider range of values for extreme events. A smaller scale parameter, as observed here, suggests that the exceedances, while heavy-tailed, are relatively concentrated around the location parameter once they cross the threshold. It essentially controls how quickly the probabilities of exceedances decrease as values get larger.


1.  **Unbounded Extremes**: The positive shape parameter (`shape_gpd > 0`) strongly suggests that raw run points in NBA games do not have a theoretical maximum. While very high runs are rare, the model indicates that even larger runs are always theoretically possible, with probabilities that diminish slowly.
2.  **Better Tail Representation**: The GPD, when applied to exceedances, provides a more accurate representation of the *extreme* tail behavior compared to a GEV model fitted to the entire distribution. This is crucial for understanding and predicting truly rare and impactful events.
3.  **Variability of Extremes**: The scale parameter quantifies the spread of these extreme runs, offering insight into how much variability there is among these large values.

In essence, while the GEV provided a first look, the GPD analysis refined our understanding of the most extreme raw runs, indicating a heavier, unbounded tail. This suggests that record-breaking runs are always a statistical possibility, although with decreasing likelihood as the run size increases.

### Q2 — Distribution des plus grands écarts de score (blowouts)
Quelle loi ajuster sur les maxima de différentiel ?
Une équipe a-t-elle tendance à répéter ce type de performances sur une saison ? (Cas symétrique : séries de défaites sévères.)